# RetailMind — Universal Retail Analytics Pipeline
**Data Science Practicum 2 · Sai Teja Sunku**

## Why this matters
Walmart, Costco and Rossmann have dedicated data teams. Independent and small retailers don't. RetailMind closes that gap: drop in any retail sales CSV/Excel and get forecasts, anomaly alerts, sales drivers, and reorder recommendations back.

## Notebook structure
1. Universal schema mapping
2. Canonicalization
3. **Feature engineering** *(addresses feedback item #2)*
4. **Hyperparameter tuning** *(addresses feedback item #3)*
5. Forecasting + walk-forward CV
6. Anomaly detection
7. Sales-driver regression (with leakage guard)
8. Order recommendations
9. **Natural-language Q&A assistant** *(addresses feedback item #5 — works offline)*
10. Static HTML report for deployment
11. Universality proven on a 2nd dataset

Datasets:
- **Rossmann** (Kaggle): daily store sales, 1.02 M rows × 1,115 stores.
- **Walmart Retail**: transactional order-line, 8.4 K rows, no explicit store column.

> **Note** — outputs in this notebook are not pre-rendered. Open it in Jupyter / VS Code / Colab and **Run All** to see results. The full Rossmann pipeline takes ~1–2 min locally.

In [ ]:
import sys, warnings, json
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from retailmind import RetailPipeline, ask, tune_lgbm, explain_grid, QUICK_GRID, save_report
from retailmind.canonical import canonicalize
from retailmind.features import (
    add_calendar_features, add_lag_features, add_rolling_features,
    add_aux_lag_features, build_feature_matrix, feature_columns,
    LEAKY_CONTEMPORANEOUS, DEFAULT_LAGS, DEFAULT_ROLLS,
)

## 1. Universal schema mapping
The mapper auto-detects which raw column plays which canonical role using name + dtype + value semantics. Every downstream module reads only canonical column names, so swapping datasets requires zero code changes.

In [ ]:
ros = RetailPipeline.from_files('../train.csv', auxiliary_paths=['../store.csv'])
print(ros.mapping.report())

## 2. Canonicalization
Raw → canonical (`date`, `entity_id`, `sales`, plus optional roles) with daily aggregation and gap-filling per entity.

In [ ]:
ros.canonicalize_()
ros.eda_()
print(json.dumps(ros.eda_report['overview'], indent=2))

## 3. Feature engineering *(addresses one of the feedback points — "no feature engineering")*

**11 helper functions in `retailmind.features`** build the matrix the ML models consume:

| Group | Features | Notes |
|---|---|---|
| Calendar | `year, month, day, dayofweek, dayofyear, weekofyear, quarter, is_weekend, is_month_start, is_month_end` | Always available |
| Cyclical | `month_sin, month_cos, dow_sin, dow_cos` | Stabilises tree boundaries near year/week wrap-around |
| Sales lags | `sales_lag_1, _lag_7, _lag_14, _lag_28` | Captures weekly + monthly autocorrelation |
| Rolling | `sales_rmean_7/14/28, sales_rstd_7/14/28` | Computed on **shifted** target — no leakage |
| Aux lags | `customers_lag_1, customers_lag_7` | Preserves traffic signal without same-day leak |
| Categorical | ordinal-encoded `entity_id`, `StoreType`, etc. | Stays leak-safe (no target encoding) |

**Leakage guard** — these columns are auto-excluded from features when target is `sales`:

In [ ]:
print('LEAKY_CONTEMPORANEOUS =', LEAKY_CONTEMPORANEOUS)
print('\nWhy:')
print('• quantity / unit_price / discount / profit → accounting identity (sales ≈ qty × price − discount + profit)')
print('• customers → same-day foot traffic. By the time you know today\'s customers, today\'s sales are realised.')
print('• Their LAGGED versions ARE allowed (yesterday\'s traffic is known today).')

In [ ]:
# Apply the feature engineering to a sampled slice and inspect what gets built
top_stores = ros.canonical.groupby('entity_id')['sales'].sum().nlargest(50).index
sample = ros.canonical[ros.canonical['entity_id'].isin(top_stores)].copy()
matrix = build_feature_matrix(sample)
feats = feature_columns(matrix)
print(f'Engineered feature matrix: {matrix.shape[0]:,} rows × {matrix.shape[1]} cols')
print(f'Usable features for the model ({len(feats)}):')
for f in feats:
    print(f'  • {f}')

## 4. Hyperparameter tuning *(addresses one of the feedback points — "no hyperparameters mentioned")*

Walk-forward CV grid search on a defensible LightGBM grid:

In [ ]:
print(explain_grid(QUICK_GRID))

In [ ]:
# Run the tuning on a 30-store sample (fast)
top30 = ros.canonical.groupby('entity_id')['sales'].sum().nlargest(30).index
df_t = ros.canonical[ros.canonical['entity_id'].isin(top30)].copy()
result = tune_lgbm(df_t, grid=QUICK_GRID, metric='rmse', n_folds=2, verbose=False)
print(f'Best CV RMSE = {result.best_score:.1f}')
print(f'Best params  = {result.best_params}')
result.all_trials

## 5. Forecasting with walk-forward CV

- Global LightGBM model trained across all entities (cross-learning).
- Walk-forward CV: 3 folds, each fold trains on `[t0, t]` and validates on `[t, t+28]`. **Random k-fold would be invalid** on time-series — that's a classic pitfall.
- Always reports a **seasonal-naïve baseline** for comparison.
- Multiple metrics: RMSE / MAE / MAPE / SMAPE (SMAPE is the most stable on retail data with many zero-sales days).

In [ ]:
ros.horizon = 28
ros.max_entities_for_full_forecast = 10
ros.forecast_(sample_entities=50, cv_folds=3)
print('Walk-forward CV metrics (mean across 3 folds):')
print(json.dumps(ros.forecast_model.cv_metrics['mean'], indent=2))

In [ ]:
top_store = ros.canonical.groupby('entity_id')['sales'].sum().idxmax()
hist = ros.canonical[ros.canonical['entity_id'] == top_store].tail(120)
fc = ros.forecast[ros.forecast['entity_id'] == top_store]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hist['date'], hist['sales'], label='history', linewidth=1)
ax.plot(fc['date'], fc['yhat'], label='forecast', linewidth=2, marker='o', markersize=3)
ax.set_title(f'Rossmann store {top_store} — 28-day forecast'); ax.legend(); plt.show()

## 6. Anomaly detection (with explanations)
Three independent detectors combined: IsolationForest, STL residual z-score, IQR. Every anomaly row carries a `reason` string explaining *why* it was flagged.

In [ ]:
ros.anomalies_(sample_entities=25)
print('Top 8 anomalies:')
print(ros.anomalies.head(8).to_string(index=False))

## 7. Sales-driver regression (with leakage guard active)
Same LightGBM, with leaky contemporaneous features (`customers`, `quantity`, `unit_price`, `discount`, `profit`) automatically excluded. Reports R², RMSE, top features + direction sign.

In [ ]:
ros.drivers_(sample_entities=50)
print(f'Holdout R² = {ros.driver_report.r2:.3f}    Holdout RMSE = {ros.driver_report.rmse:,.1f}')
print('\nTop 10 drivers (after leakage exclusion):')
print(ros.driver_report.importance.head(10)[['feature', 'importance', 'direction_label']].to_string(index=False))

## 8. Order recommendations
Forecast → reorder point: `ROP = mean × lead_time + z(SL) × std × √lead_time`. User picks service level and lead time.

In [ ]:
ros.recommendations_()
print('Top 10 reorder recommendations:')
print(ros.recommendations.head(10).to_string(index=False))

## 9. Natural-language Q&A *(addresses one of the feedback points — chatbot)*
Works **offline**, no API key needed. Returns specific numbers from the pipeline outputs.

In [ ]:
for q in [
    'Give me an overview of the dataset',
    'How accurate is the forecast model?',
    'What drives sales the most?',
    'Show me the top anomalies',
    'How much should I order?',
    'How does promo affect sales?',
    "What's the day-of-week pattern?",
]:
    print('-' * 70)
    print('Q:', q)
    print(ask(ros, q, mode='offline'))
    print()

## 10. Static HTML report
Renders the full pipeline output into a self-contained `index.html` that can be served from GitHub Pages / Netlify / any static host.

In [ ]:
path = save_report(ros, out_dir='../docs', filename='rossmann.html', dataset_name='Rossmann Store Sales')
print(f'Report → {path}  ({path.stat().st_size:,} bytes, fully self-contained)')
print('Open in a browser, or push the docs/ folder and enable GitHub Pages from /docs.')

## 11. Same pipeline on Walmart — proves universality
**Identical lines of code** as above, but on a completely different dataset (order-line transactional, no store column).

In [ ]:
wal = RetailPipeline.from_files('../walmart Retail Data.xlsx')
print(wal.mapping.report())
print()
wal.horizon = 28
wal.max_entities_for_full_forecast = 10
wal.run(cv_folds=3)
print(f'Walmart EDA: {wal.eda_report["overview"]["entities"]} entity, '
      f'{wal.eda_report["overview"]["rows"]:,} rows')
print(f'Walmart Forecast SMAPE: {wal.forecast_model.cv_metrics["mean"]["smape"]:.1f}%')
print(f'Walmart Driver R²: {wal.driver_report.r2:.3f}  (low because transactional data + single global entity is genuinely hard)')
_ = save_report(wal, out_dir='../docs', filename='walmart.html', dataset_name='Walmart Retail Data')
print('Walmart report written to ../docs/walmart.html')

## Summary — addressing every v1 feedback item

| Feedback | v2 response |
|---|---|
| #1 dataset outdated | The pipeline is dataset-agnostic. Any current dataset uploaded to the Streamlit app or passed to `RetailPipeline.from_files()` runs the full stack. |
| #2 no feature engineering | Section **3** above: 11 helpers, leakage guard, lag/rolling, cyclical encodings. |
| #3 hyperparameters not mentioned | Section **4**: explicit grid + rationale + best params. |
| #4 slides | See `SLIDES.md` in the repo root. |
| #5 chatbot not implemented | Section **9**: offline assistant, works without API key. |
| #6 single notebook | This notebook + `retailmind/` package (11 modules) + `app.py` + static report + `tests/test_universal.py`. |
| Deployment | Static report (Section 10) deploys to GitHub Pages / Netlify. Streamlit app deploys to share.streamlit.io. See `DEPLOYMENT.md`. |